# Week 72, Day 2: Mocking

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w072d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from unittest.mock import Mock

api = Mock()
api.get_user.return_value = {"name": "Ada"}

print(api.get_user(1)["name"])
print(api.get_user.call_count, api.get_user.call_args)
print(type(api.delete_everything()).__name__)
print(api.method_calls)

In [ ]:
from unittest.mock import Mock, MagicMock, call

payments = Mock()
payments.charge.side_effect = [ConnectionError("timeout"), {"status": "ok"}]

for attempt in (1, 2):
    try:
        print(attempt, payments.charge("ada", 25, currency="GBP"))
    except ConnectionError as e:
        print(attempt, "failed:", e)

payments.charge.assert_called_with("ada", 25, currency="GBP")
print(payments.charge.call_count, payments.charge.call_args_list == [call("ada", 25, currency="GBP")] * 2)

rates = Mock(side_effect=lambda code: {"GBP": 1.0, "EUR": 1.17}[code])
print(rates("EUR"))
try:
    rates("XYZ")
except KeyError as e:
    print("KeyError", e)

box = MagicMock()
box.__len__.return_value = 3
box.__getitem__.side_effect = lambda i: i * 10
print(len(box), box[4])
with box as inside:
    print(inside is box.__enter__.return_value)

In [ ]:
import sys, types
from unittest.mock import patch

clock = types.ModuleType("clock")
exec("from time import time\n\ndef stamp(event):\n    return f'{event} at {time():.0f}'\n", clock.__dict__)
sys.modules["clock"] = clock

with patch("time.time", return_value=1000):
    print(clock.stamp("wrong place"))       # clock has its own name 'time': not patched

with patch("clock.time", return_value=1000) as fake:
    print(clock.stamp("right place"))
    print(fake.call_count)

print(clock.stamp("afterwards") != "afterwards at 1000")

In [ ]:
from unittest.mock import Mock, create_autospec

class Mailer:
    def send(self, to, subject, body=""):
        ...

def welcome(user, mailer):
    mailer.sned(user, "Welcome!")        # a typo in the method name

loose = Mock()
welcome("ada@example.com", loose)
print("loose mock: the typo goes unnoticed")

strict = create_autospec(Mailer, instance=True)
try:
    welcome("ada@example.com", strict)
except AttributeError as e:
    print("autospec:", e)

try:
    strict.send("ada@example.com")       # missing the subject
except TypeError as e:
    print("autospec:", e)

> **Key idea.** Test doubles stand in for collaborators at the edges: stubs answer, mocks record, spies wrap, fakes are simple working versions. Mock invents attributes, accepts any call and records it; return_value sets the answer; side_effect raises an exception, steps through an iterable or calls a function. Check calls with assert_called_once_with, assert_has_calls and call_args_list; MagicMock adds special methods. patch replaces a name for a block or a test and restores it afterwards: patch where the name is looked up. spec and create_autospec make mocks reject calls the real object wouldn't accept. Mock little, check the calls that are the behaviour, and prefer fakes for stateful things.

### Exercise: A programmed stand-in

Write `make_directory(users)`: return a `Mock` standing in for a user service, configured so that `service.lookup(user_id)` returns the user's dict from `users` (a dict keyed by id) and raises `KeyError` for an unknown id, and `service.count()` returns how many users there are. Then `lookups(service)`: the list of ids that `lookup` has been called with so far, in order, read from the mock's records.

In [ ]:
from unittest.mock import Mock

def make_directory(users):
    ___

def lookups(service):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_directory():\n    \"return_value, side_effect and call records\"\n    svc = make_directory({1: {\"name\": \"Ada\"}, 2: {\"name\": \"Bob\"}})\n    assert svc.lookup(2)[\"name\"] == \"Bob\" and svc.count() == 2\n    try:\n        svc.lookup(9)\n        assert False\n    except KeyError:\n        pass\n    svc.lookup(1)\n    assert lookups(svc) == [2, 9, 1]\n    assert lookups(make_directory({})) == []")

### Exercise: Checking the calls that matter

`reminders.py` has `send_reminders(members, mailer, today)`: for each member whose `"renewal"` date is within 7 days from `today` (inclusive, and not in the past), it calls `mailer.send(member["email"], "Your membership renews soon")` once, and it returns how many it sent. Write tests that pass a `Mock` as the mailer, check the calls it received, and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('reminders.py', 'w', encoding='utf-8') as f:
    f.write('from datetime import timedelta\n\ndef send_reminders(members, mailer, today):\n    sent = 0\n    for member in members:\n        days = (member["renewal"] - today).days\n        if 0 <= days <= 7:\n            mailer.send(member["email"], "Your membership renews soon")\n            sent += 1\n    return sent\n')

In [ ]:
from datetime import date
from unittest.mock import Mock, call
from reminders import send_reminders

def test_reminders():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"reminders.py\").read()\nBROKEN = {\n    \"sends to everyone\": GOOD.replace(\"        if 0 <= days <= 7:\\n\", \"        if True:\\n\"),\n    \"misses the seventh day\": GOOD.replace(\"0 <= days <= 7\", \"0 <= days < 7\"),\n    \"reminds about past renewals\": GOOD.replace(\"0 <= days <= 7\", \"days <= 7\"),\n    \"wrong subject\": GOOD.replace('\"Your membership renews soon\"', '\"Renew now\"'),\n    \"sends twice\": GOOD.replace('            mailer.send(member[\"email\"], \"Your membership renews soon\")\\n', '            mailer.send(member[\"email\"], \"Your membership renews soon\")\\n            mailer.send(member[\"email\"], \"Your membership renews soon\")\\n'),\n    \"wrong count\": GOOD.replace(\"    return sent\", \"    return len(members)\"),\n}\ndef test_catches():\n    \"your tests pass on the correct code and catch all six broken versions\"\n    results = against(source, \"reminders\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"reminders.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Failing on purpose

`fetching.py` has `fetch_with_retry(client, url, attempts=3)`: it calls `client.get(url)` up to `attempts` times while it raises `ConnectionError`, returning the first result; after the last failure it re-raises; other exceptions propagate at once without retrying. Write tests using a `Mock` client whose `get` has a `side_effect` **list**, and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('fetching.py', 'w', encoding='utf-8') as f:
    f.write('def fetch_with_retry(client, url, attempts=3):\n    for attempt in range(1, attempts + 1):\n        try:\n            return client.get(url)\n        except ConnectionError:\n            if attempt == attempts:\n                raise\n')

In [ ]:
import pytest
from unittest.mock import Mock
from fetching import fetch_with_retry

def test_recovers():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"fetching.py\").read()\nBROKEN = {\n    \"one attempt too few\": GOOD.replace(\"range(1, attempts + 1)\", \"range(1, attempts)\"),\n    \"swallows the last error\": GOOD.replace(\"            if attempt == attempts:\\n                raise\\n\", \"            pass\\n\"),\n    \"retries every error\": GOOD.replace(\"except ConnectionError:\", \"except Exception:\"),\n    \"forgets the url\": GOOD.replace(\"client.get(url)\", \"client.get()\"),\n}\ndef test_catches():\n    \"your tests pass on the correct code and catch all four broken versions\"\n    results = against(source, \"fetching\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"fetching.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Patching the right name

`forecast.py` does `from weather_api import get_temperature` and has `advice(city)`, which returns `"coat"` below 10 degrees, `"jumper"` from 10 up to (not including) 18, and `"t-shirt"` from 18. The real `get_temperature` calls the network (here, it raises). Write tests that use `unittest.mock.patch` on the **right name** to give fixed temperatures, and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('weather_api.py', 'w', encoding='utf-8') as f:
    f.write('def get_temperature(city):\n    raise RuntimeError("no network in tests")\n')
with open('forecast.py', 'w', encoding='utf-8') as f:
    f.write('from weather_api import get_temperature\n\ndef advice(city):\n    t = get_temperature(city)\n    if t < 10:\n        return "coat"\n    if t < 18:\n        return "jumper"\n    return "t-shirt"\n')

In [ ]:
from unittest.mock import patch
from forecast import advice

def test_cold():
    with patch("weather_api.get_temperature", return_value=5):
        assert advice("Leeds") == "coat"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"forecast.py\").read()\nBROKEN = {\n    \"10 is a coat\": GOOD.replace(\"if t < 10:\", \"if t <= 10:\"),\n    \"18 is a jumper\": GOOD.replace(\"if t < 18:\", \"if t <= 18:\"),\n    \"ignores the city\": GOOD.replace(\"get_temperature(city)\", 'get_temperature(\"London\")'),\n}\ndef test_catches():\n    \"your tests patch forecast's own name and catch all three broken versions\"\n    results = against(source, \"forecast\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"forecast.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code (did you patch the right name?): {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: A mock that knows the real class

`signup.py` has `register(email, store, mailer)`. Its tests used plain mocks, so a bug slipped through: one broken version calls `mailer.send_welcome(email)` (the real `Mailer` method is `send(to, subject)`), and another calls `store.save(email, True)` (the real method takes one argument). Write tests that use **`create_autospec`** for both collaborators, so that both broken versions fail, and check that the email was saved and the right message sent.

In [ ]:
# Run this cell first: it creates the files used below.
with open('services.py', 'w', encoding='utf-8') as f:
    f.write('class Store:\n    def save(self, email):\n        raise RuntimeError("real database")\n\n    def exists(self, email):\n        raise RuntimeError("real database")\n\nclass Mailer:\n    def send(self, to, subject):\n        raise RuntimeError("real email")\n')
with open('signup.py', 'w', encoding='utf-8') as f:
    f.write('def register(email, store, mailer):\n    if store.exists(email):\n        return False\n    store.save(email)\n    mailer.send(email, "Welcome to Atlas")\n    return True\n')

In [ ]:
from unittest.mock import Mock, create_autospec
from services import Store, Mailer
from signup import register

def test_register():
    store, mailer = Mock(), Mock()
    store.exists.return_value = False
    assert register("ada@x", store, mailer) is True

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"signup.py\").read()\nBROKEN = {\n    \"a method that doesn't exist\": GOOD.replace('mailer.send(email, \"Welcome to Atlas\")', \"mailer.send_welcome(email)\"),\n    \"the wrong arguments\": GOOD.replace(\"store.save(email)\", \"store.save(email, True)\"),\n    \"saves an existing user\": GOOD.replace(\"        return False\\n\", \"        pass\\n\"),\n    \"no email\": GOOD.replace('    mailer.send(email, \"Welcome to Atlas\")\\n', \"\"),\n}\ndef test_catches():\n    \"autospecced doubles catch all four broken versions\"\n    assert uses(\"create_autospec\"), \"use create_autospec\"\n    results = against(source, \"signup\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"signup.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: A fake instead of a mock

`ShortLinks` (given) creates short codes for URLs and stores them in a **storage** object with methods `get(key)` (returning `None` for a missing key), `put(key, value)` and `keys()`. Write `FakeStorage`, a working in-memory version backed by a dict, that also counts `puts` (how many times `put` was called). Then use it: `make_links()` returns a `ShortLinks` using a new `FakeStorage`.

In [ ]:
class ShortLinks:
    def __init__(self, storage):
        self.storage = storage

    def shorten(self, url):
        for key in self.storage.keys():
            if self.storage.get(key) == url:
                return key
        key = f"s{len(list(self.storage.keys())) + 1}"
        self.storage.put(key, url)
        return key

    def expand(self, key):
        url = self.storage.get(key)
        if url is None:
            raise KeyError(key)
        return url

class FakeStorage:
    ___

def make_links():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fake():\n    \"a working fake\"\n    links = make_links()\n    a = links.shorten(\"https://example.com/a\")\n    b = links.shorten(\"https://example.com/b\")\n    assert (a, b) == (\"s1\", \"s2\") and links.shorten(\"https://example.com/a\") == \"s1\"\n    assert links.expand(\"s2\") == \"https://example.com/b\" and links.storage.puts == 2\n    try:\n        links.expand(\"s9\")\n        assert False\n    except KeyError:\n        pass\n    s = FakeStorage()\n    assert s.get(\"x\") is None and list(s.keys()) == []\n    s.put(\"x\", 1)\n    assert s.get(\"x\") == 1 and list(s.keys()) == [\"x\"] and s.puts == 1")

### Bug hunt: Patched in the wrong place

`stamps.py` does `from time import time`, so patching `"time.time"` doesn't affect it, and the test gets the real clock and fails. Patch the name `stamps` actually uses so the test passes.

In [ ]:
# Run this cell first: it creates the files used below.
with open('stamps.py', 'w', encoding='utf-8') as f:
    f.write('from time import time\n\ndef stamp(event):\n    return f"{event} at {time():.0f}"\n')

In [ ]:
from unittest.mock import patch
from stamps import stamp

def test_stamp():
    with patch("time.time", return_value=1000):
        assert stamp("login") == "login at 1000"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"the clock is patched where it's looked up\"\n    r = pytest_run(source)\n    assert r[\"passed\"] == [\"test_stamp\"], r[\"output\"][-600:]")

### Bug hunt: A check that checks nothing

`mailer.send.called_once_with(...)` isn't an assertion: it's just another automatically created mock attribute, called and ignored, so this test passes even on the broken version that sends nothing. Use the real assertion method so that the broken version is caught.

In [ ]:
# Run this cell first: it creates the files used below.
with open('greet.py', 'w', encoding='utf-8') as f:
    f.write('def greet(user, mailer):\n    mailer.send(user, "Hello")\n')

In [ ]:
from unittest.mock import Mock
from greet import greet

def test_greet():
    mailer = Mock()
    greet("ada@x", mailer)
    mailer.send.called_once_with("ada@x", "Hello")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"the broken version is caught\"\n    good = open(\"greet.py\").read()\n    results = against(source, \"greet\", {\"correct\": good, \"sends nothing\": \"def greet(user, mailer):\\n    pass\\n\"})\n    write_file(\"greet.py\", good)\n    assert results[\"correct\"] == [] and results[\"sends nothing\"], results")

### Bug hunt: A side effect that should have been a return value

`side_effect` set to a dict is treated as an **iterable**: the first call returns the dict's first key, `"name"`, so `profile` fails. Set the answer with `return_value` so this prints `Ada (36)`.

In [ ]:
from unittest.mock import Mock

def profile(api, user_id):
    user = api.get_user(user_id)
    return f"{user['name']} ({user['age']})"

api = Mock()
api.get_user.side_effect = {"name": "Ada", "age": 36}
print(profile(api, 1))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a return value\"\n    expect_output(\"Ada (36)\")")

### Puzzle: Patching for one block

Arrange the lines of a test that patches a module's random choice, checks the result, and checks the mock was called with the list of options. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # m = Mock(); m(1); m(2); m.call_count
q = ___   # m = Mock(side_effect=[1, 2]); m(); m()   (the second call's result)
r = ___   # True or False: Mock(spec=list).append exists and Mock(spec=list).push raises AttributeError
s = ___   # m = Mock(return_value=5); m.side_effect = lambda: 9; m()
t = ___   # the name to patch when shop.py does "from random import choice": "random.choice" or "shop.choice"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 2 and r is True, \"p, q and r\"\n    assert s == 9 and t == \"shop.choice\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a mock of your own

Write a small `Double` class that works like `Mock`. Getting any attribute that doesn't start with `_` creates (and remembers) a child `Double` named after it. Calling a `Double` records the call as `(args, kwargs)` in its `calls` list, and also appends `(name, args, kwargs)` to the **root** double's `log`, where `name` is the dotted path from the root (`"api.get"` for `root.api.get(...)`, `""` for the root itself). The result is decided by `side_effect` if set (an exception class or instance is raised; a function is called with the same arguments; any other iterable gives its next item per call, and an exception item is raised), otherwise `return_value` (which defaults to a new child `Double` named `"()"`, the same one every time). Also `call_count` (a property), `assert_called_once_with(*args, **kwargs)` (raising `AssertionError` with a message unless there was exactly one call, with those arguments) and `reset()` (clearing its calls, and the log if it's the root).

In [ ]:
class Double:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_double():\n    \"attributes, calls, results and the log\"\n    root = Double()\n    assert root.api is root.api and isinstance(root.api.get, Double)\n    root.api.get.return_value = {\"id\": 1}\n    assert root.api.get(1, full=True) == {\"id\": 1} and root.api.get.calls == [((1,), {\"full\": True})]\n    root.api.post(\"x\")\n    assert root.log == [(\"api.get\", (1,), {\"full\": True}), (\"api.post\", (\"x\",), {})]\n    root.api.get.assert_called_once_with(1, full=True)\n    try:\n        root.api.post.assert_called_once_with(\"y\")\n        assert False\n    except AssertionError:\n        pass\n    assert root() is root() and isinstance(root(), Double) and root.call_count == 3\n    flaky = Double()\n    flaky.side_effect = [ValueError(\"no\"), 5]\n    try:\n        flaky()\n        assert False\n    except ValueError:\n        pass\n    assert flaky() == 5\n    flaky.side_effect = lambda x: x * 2\n    assert flaky(4) == 8\n    flaky.side_effect = KeyError\n    try:\n        flaky()\n        assert False\n    except KeyError:\n        pass\n    root.reset()\n    assert root.calls == [] and root.log == []")

## Warm-up

A `Mock` invents any attribute you ask for and accepts any call. That makes it endlessly convenient, and it's also its biggest danger: a mock will happily stand in for a method the real object doesn't have.

## Test doubles

A **test double** is anything that stands in for a real collaborator in a test. The names are worth knowing, because they describe different jobs:

- a **stub** returns canned answers (`get_rate()` always returns `1.2`), so the code under test can run;
- a **mock** records how it was called, so the test can check the code made the right calls (`send` was called once, with this address);
- a **spy** wraps the real object, passing calls through while recording them;
- a **fake** is a simple working implementation: an in-memory dictionary instead of a database, a list instead of an email server.

Doubles belong at the **edges** of a system: things that are slow (networks, databases), non-deterministic (time, randomness), or have side effects you can't allow (payments, emails). Code in the middle, your own pure functions and classes, should usually be tested for real. Week 24 Day 3 made code testable by **passing in** the clock and the random generator; that's still the cleanest option, and doubles are what you pass in.

## Mock objects

`unittest.mock.Mock` creates an object that:

- creates child mocks on demand: `m.anything` is a `Mock`, and the same one each time;
- accepts any call, returning `m.return_value` (itself a `Mock` unless you set it);
- records every call: `called`, `call_count`, `call_args` (the last call), `call_args_list` (all of them), and `mock_calls` / `method_calls` on the parent;
- runs a **`side_effect`** if one is set: an **exception** (class or instance) is raised; an **iterable** gives one result per call (an exception in it is raised on that call); a **function** is called with the same arguments and its result returned.

It also has assertion methods: `assert_called()`, `assert_called_once()`, `assert_called_with(...)` (the **last** call), `assert_called_once_with(...)`, `assert_any_call(...)`, `assert_has_calls([call(...), ...])` and `assert_not_called()`. `unittest.mock.call(...)` describes one expected call.

**`MagicMock`** is a `Mock` that also supports the special methods (`len(m)`, `m[0]`, `with m:`, iteration), each configurable: `m.__len__.return_value = 3`.

## patch, and where to patch

Often the collaborator isn't passed in: the code calls a function from another module directly. **`unittest.mock.patch(target)`** replaces the object at `target` (a dotted name, as a string) for the length of a `with` block, a decorated test function, or until you call `stop()`, and then puts the original back, even if the test fails.

The rule that catches everyone: **patch the name where it is looked up, not where it's defined.**

- If `weather.py` does `import requests` and calls `requests.get(...)`, it looks up `get` on the `requests` module each time, so patch `"requests.get"` (or `"weather.requests.get"`, the same object).
- If `weather.py` does `from requests import get` and calls `get(...)`, it has its **own** name `get`, bound at import. Patching `"requests.get"` changes the module's attribute but not `weather`'s name, so the real function still runs. Patch `"weather.get"`.

Variants: `patch.object(obj, "attribute")` patches an attribute of an object you already have; `patch.dict(os.environ, {...})` changes a dictionary temporarily; and pytest's `monkeypatch.setattr` (Week 24 Day 5) does the same job as `patch` in fixture form. `patch` passes the new mock to the `with` block (`as m`) or to the decorated function as an extra argument.

## Specs, and when not to mock

A bare `Mock` accepts **anything**: misspell a method name in your code, call it with the wrong arguments, and the test still passes. **Specs** fix that:

- `Mock(spec=RealClass)` only allows attributes the real class has (getting any other raises `AttributeError`).
- **`create_autospec(RealClass)`** (or `patch(..., autospec=True)`) goes further: methods check their **arguments** against the real signatures, so a call the real object would reject fails the test too.

Use autospec by default when mocking your own classes. And then use mocks less than you think:

- **Don't mock what you don't own** directly: wrap a third-party library in a small class of your own, test that thin layer against the real thing occasionally, and mock or fake **your** wrapper elsewhere.
- **Don't assert on every call.** Tests that check the exact sequence of internal calls break whenever the code is reorganised, even when its behaviour is unchanged. Check the calls that **are** the behaviour (the email was sent, the payment was charged once), and check results otherwise.
- **Prefer fakes for stateful collaborators.** A fake repository built on a dictionary lets tests check what was stored, by reading it back, rather than by checking which methods were called.
- **A test that only exercises mocks tests nothing.** If every collaborator is a mock and every check is about the mocks, the test proves the code calls what it calls.

## Practice

## On your own computer

pytest's `monkeypatch` and `unittest.mock.patch` work the same on your computer, but there's one library worth adding for mocks in pytest: **pytest-mock** gives a `mocker` fixture that undoes every patch at the end of the test, with no `with` blocks:

```python
def test_advice(mocker):
    fake = mocker.patch("forecast.get_temperature", return_value=12)
    assert advice("Leeds") == "jumper"
    fake.assert_called_once_with("Leeds")
```

Install it with `pip install pytest-mock`. For HTTP specifically, libraries such as `responses` (for `requests`) and `respx` (for `httpx`) fake the server side, so your code's real HTTP calls get canned replies: closer to reality than mocking the client.

## Quiz

## Daily challenge

## Before you go

Mocks are a sharp tool: they make the impossible tests possible and the easy tests fragile. Tomorrow: **property-based testing**, where instead of choosing examples, you describe rules and let the computer search for the inputs that break them.